In [0]:
def get_cineclub():
    df = spark.sql(
        """
        SELECT 
            DATE_SUB(DATE(REPORTING_DATETIME), (DAYOFWEEK(REPORTING_DATETIME) + 1) % 7) AS `week`, 
            SUM(CASE WHEN PLAN_ID = 'monthly1' THEN 1 ELSE 0 END) AS monthly_signups,
            SUM(CASE WHEN PLAN_ID = 'annual1' THEN 1 ELSE 0 END) AS annual_signups,
            SUM(CASE WHEN PLAN_ID = 'gift-plan' THEN 1 ELSE 0 END) AS gift_signups
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        WHERE EVENT_TYPE = 'subscription_created'
        AND STATUS = 'active'
        AND REPORTING_DATETIME IS NOT NULL
        AND REPORTING_DATETIME BETWEEN '2023-03-31' AND '2026-02-28'
        GROUP BY `week`
        ORDER BY `week`
        """
    ).toPandas()

    return df

In [0]:
cineclub = get_cineclub() # Backup in 'cineclub_signups.csv'

In [0]:
display(cineclub)

In [0]:
def get_releases():
    df = spark.sql(
        """
        SELECT 
            DATE_SUB(RELEASE_DATE, (DAYOFWEEK(RELEASE_DATE) + 1) % 7) AS `week`,
            COUNT(TITLE_ID) AS films_released
        FROM (
            SELECT 
                TITLE_ID, 
                TITLE_NAME,
                MIN(THEATRICAL_RELEASE_DATE) AS RELEASE_DATE
            FROM cpx_dataanalytics_gold.customer360.cpx_d_film
            WHERE PERFORMANCE_TYPE = 'Film Presentation'
            AND THEATRICAL_RELEASE_DATE BETWEEN '2023-03-31' AND '2025-04-03'
            AND NOT UPPER(Film_Title) RLIKE 'FAN EVENT|EARLY ACCESS|FAN FIRST'
            GROUP BY TITLE_ID, TITLE_NAME
        ) AS R
        GROUP BY `week`
        ORDER BY `week`
        """
    ).toPandas()

    return df

In [0]:
releases = get_releases()

In [0]:
releases.display()

In [0]:
%sql
SELECT
    CASE
        WHEN recognitions_left = 0        THEN '0'
        WHEN recognitions_left = 1        THEN '1'
        WHEN recognitions_left BETWEEN 2 AND 3  THEN '2 to 3'
        WHEN recognitions_left BETWEEN 4 AND 5  THEN '4 to 5'
        WHEN recognitions_left BETWEEN 5 AND 7  THEN '6 to 7'
        WHEN recognitions_left BETWEEN 8 AND 10 THEN '8 to 10'
        WHEN recognitions_left > 10       THEN '10+'
        ELSE 'Other'
    END AS recognitions_bucket,
    COUNT(DISTINCT CDE_ID) AS member_count
FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
WHERE snapshot_date_id = '20260314'
GROUP BY recognitions_bucket
ORDER BY MIN(recognitions_left);

In [0]:
%sql
SELECT
    CASE
        WHEN recognitions_left = 0          THEN '0'
        WHEN recognitions_left = 1               THEN '1'
        WHEN recognitions_left = 2               THEN '2'
        WHEN recognitions_left = 3               THEN '3'
        WHEN recognitions_left = 4               THEN '4'
        WHEN recognitions_left = 5               THEN '5'
        WHEN recognitions_left = 6               THEN '6'
        WHEN recognitions_left = 7               THEN '7'
        WHEN recognitions_left = 8               THEN '8'
        WHEN recognitions_left = 9               THEN '9'
        WHEN recognitions_left >= 10              THEN '10+'
        ELSE 'Other'
    END AS recognitions_bucket,
    COUNT(DISTINCT CDE_ID)                                                        AS total_members,
    COUNT(DISTINCT CASE WHEN renewal_status LIKE '%Active%'    THEN CDE_ID END)   AS active_members,
    COUNT(DISTINCT CASE WHEN renewal_status LIKE 'Cancelled%'  THEN CDE_ID END)   AS cancelled_members,
    COUNT(DISTINCT CASE WHEN renewal_status NOT LIKE '%Active%' 
                         AND renewal_status NOT LIKE 'Cancelled%' THEN CDE_ID END) AS other_status_members
FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
WHERE snapshot_date_id = '20260314'
GROUP BY recognitions_bucket
ORDER BY MIN(recognitions_left);